In [1]:

import os
import glob
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

# ----------------------------------------------------------------------
# Required annotation columns
# ----------------------------------------------------------------------

REQUIRED_COLUMNS = [
    "fhr",
    "uc",
    "fhr_nan",
    "uc_nan",
    "baseline",
    "acc",
    "dec",
]


# ----------------------------------------------------------------------
# Find .csv.csv annotation files
# ----------------------------------------------------------------------

drive.mount("/content/drive")


BASE_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Fetal_project"
)

ANO_DIR = os.path.join(
    BASE_DIR,
    "CTGDL_FHRMA_ano_csv"
)

csv_files = sorted(
    glob.glob(
        os.path.join(ANO_DIR, "*.csv.csv")
    )
)


print("\n" + "=" * 80)
print("ANNOTATION DATASET SELECTION")
print("=" * 80)

print(
    f"Found .csv.csv files: {len(csv_files)}"
)


if len(csv_files) == 0:

    raise FileNotFoundError(
        f"No .csv.csv annotation files found in:\n{ANO_DIR}"
    )


# ----------------------------------------------------------------------
# Check every annotation file
# ----------------------------------------------------------------------

valid_annotation_files = []
invalid_annotation_files = []


for filepath in csv_files:

    filename = os.path.basename(filepath)

    try:

        # Only read the header for this check
        df = pd.read_csv(
            filepath,
            nrows=5
        )

        missing_columns = [
            column
            for column in REQUIRED_COLUMNS
            if column not in df.columns
        ]

        if len(missing_columns) == 0:

            valid_annotation_files.append(filepath)

        else:

            invalid_annotation_files.append(
                (
                    filename,
                    missing_columns
                )
            )

    except Exception as e:

        invalid_annotation_files.append(
            (
                filename,
                [f"READ ERROR: {e}"]
            )
        )


# ----------------------------------------------------------------------
# Display invalid files, if any
# ----------------------------------------------------------------------

if len(invalid_annotation_files) > 0:

    print("\nInvalid annotation files:")

    for filename, problem in invalid_annotation_files:

        print(
            f"  {filename}: {problem}"
        )


# ----------------------------------------------------------------------
# Stop if no valid files
# ----------------------------------------------------------------------

if len(valid_annotation_files) == 0:

    raise RuntimeError(
        "No valid annotation files were found."
    )


# ----------------------------------------------------------------------
# Final annotation file list
# ----------------------------------------------------------------------

ANNOTATION_FILES = sorted(
    valid_annotation_files
)


# ----------------------------------------------------------------------
# Final report
# ----------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL ANNOTATION DATASET")
print("=" * 80)

print(
    f"Files found:       {len(csv_files)}"
)

print(
    f"Files selected:    {len(ANNOTATION_FILES)}"
)

print(
    f"Files rejected:    {len(invalid_annotation_files)}"
)

print("\nRequired columns:")

for column in REQUIRED_COLUMNS:

    print(
        f"  ✓ {column}"
    )


print("\nFirst selected files:")

for filepath in ANNOTATION_FILES[:20]:

    print(
        f"  {os.path.basename(filepath)}"
    )

if len(ANNOTATION_FILES) > 20:

    print("  ...")


print("\n✓ Annotation file selection completed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

ANNOTATION DATASET SELECTION
Found .csv.csv files: 134

FINAL ANNOTATION DATASET
Files found:       134
Files selected:    134
Files rejected:    0

Required columns:
  ✓ fhr
  ✓ uc
  ✓ fhr_nan
  ✓ uc_nan
  ✓ baseline
  ✓ acc
  ✓ dec

First selected files:
  test01.csv.csv
  test02.csv.csv
  test04.csv.csv
  test05.csv.csv
  test06.csv.csv
  test07.csv.csv
  test08.csv.csv
  test09.csv.csv
  test11.csv.csv
  test13.csv.csv
  test14.csv.csv
  test15.csv.csv
  test16.csv.csv
  test17.csv.csv
  test18.csv.csv
  test19.csv.csv
  test20.csv.csv
  test22.csv.csv
  test23.csv.csv
  test24.csv.csv
  ...

✓ Annotation file selection completed.


In [2]:



# ══════════════════════════════════════════════════════════════════════════════
# 2. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════════

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# ══════════════════════════════════════════════════════════════════════════════
# 3. GOOGLE DRIVE + PROJECT PATHS
# ══════════════════════════════════════════════════════════════════════════════

drive.mount("/content/drive")


BASE_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Fetal_project"
)


ANO_DIR = os.path.join(
    BASE_DIR,
    "CTGDL_FHRMA_ano_csv"
)


if not os.path.isdir(BASE_DIR):

    raise FileNotFoundError(
        f"Base directory was not found:\n{BASE_DIR}"
    )


if not os.path.isdir(ANO_DIR):

    raise FileNotFoundError(
        f"Annotation directory was not found:\n{ANO_DIR}"
    )


print("=" * 80)
print("PROJECT INITIALIZATION")
print("=" * 80)

print(
    f"Base directory:\n"
    f"  {BASE_DIR}"
)

print(
    f"\nAnnotation directory:\n"
    f"  {ANO_DIR}"
)


# ══════════════════════════════════════════════════════════════════════════════
# 4. DEVICE
# ══════════════════════════════════════════════════════════════════════════════

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("\n" + "=" * 80)
print("COMPUTING DEVICE")
print("=" * 80)

print(
    f"Using device: {device}"
)

if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

else:

    print(
        "CUDA GPU not available. Using CPU."
    )

print("=" * 80)


# ══════════════════════════════════════════════════════════════════════════════
# 5. DATA / EXPERIMENT PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

FS = 4

MAX_ALLOWABLE_NAN_PCT = 0.60

VALIDATION_FRACTION = 0.20


# Two windowing experiments
EXPERIMENTS = {

    "15min_5min": {

        "window_minutes": 15,

        "stride_minutes": 5,
    },

    #"7.5min_2.5min": {

    #    "window_minutes": 7.5,

    #    "stride_minutes": 2.5,
    #},
}


# Required annotation columns
REQUIRED_COLUMNS = [

    "fhr",

    "uc",

    "fhr_nan",

    "uc_nan",

    "baseline",

    "acc",

    "dec",
]


# ══════════════════════════════════════════════════════════════════════════════
# 6. VERIFY EXPERIMENT CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("EXPERIMENT CONFIGURATION")
print("=" * 80)

print(
    f"Sampling frequency: "
    f"{FS} Hz"
)

print(
    f"Maximum allowable missingness: "
    f"{MAX_ALLOWABLE_NAN_PCT:.0%}"
)

print(
    f"Validation fraction: "
    f"{VALIDATION_FRACTION:.0%}"
)

print(
    f"Random seed: "
    f"{RANDOM_SEED}"
)


print("\nRequired columns:")

for column in REQUIRED_COLUMNS:

    print(
        f"  ✓ {column}"
    )


print("\nExperiments:")

for experiment_name, config in EXPERIMENTS.items():

    window_size = int(
        config["window_minutes"]
        * 60
        * FS
    )

    stride_size = int(
        config["stride_minutes"]
        * 60
        * FS
    )

    print(
        f"\n  {experiment_name}"
    )

    print(
        f"    Window: "
        f"{config['window_minutes']} min "
        f"= {window_size:,} samples"
    )

    print(
        f"    Stride: "
        f"{config['stride_minutes']} min "
        f"= {stride_size:,} samples"
    )


# ══════════════════════════════════════════════════════════════════════════════
# 7. VERIFY ANNOTATION FILES
# ══════════════════════════════════════════════════════════════════════════════

annotation_files = sorted(
    glob.glob(
        os.path.join(
            ANO_DIR,
            "*.csv.csv"
        )
    )
)


print("\n" + "=" * 80)
print("ANNOTATION FILE VERIFICATION")
print("=" * 80)

print(
    f"Found .csv.csv annotation files: "
    f"{len(annotation_files)}"
)


if len(annotation_files) == 0:

    raise RuntimeError(
        "No .csv.csv annotation files were found."
    )


# Check that every annotation file contains
# all required columns.
invalid_annotation_files = []

for path in annotation_files:

    filename = os.path.basename(path)

    try:

        df = pd.read_csv(path, nrows=5)

        missing_columns = [
            column
            for column in REQUIRED_COLUMNS
            if column not in df.columns
        ]

        if missing_columns:

            invalid_annotation_files.append(
                (
                    filename,
                    missing_columns
                )
            )

    except Exception as e:

        invalid_annotation_files.append(
            (
                filename,
                [f"READ ERROR: {str(e)}"]
            )
        )


print(
    f"Valid annotation files: "
    f"{len(annotation_files) - len(invalid_annotation_files)}"
)

print(
    f"Invalid annotation files: "
    f"{len(invalid_annotation_files)}"
)


if invalid_annotation_files:

    print("\nInvalid files:")

    for filename, problems in invalid_annotation_files:

        print(
            f"  {filename}: {problems}"
        )

    raise RuntimeError(
        "Some annotation files failed verification."
    )


# ══════════════════════════════════════════════════════════════════════════════
# 8. FINAL CONFIGURATION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("✓ INITIALIZATION COMPLETED")
print("=" * 80)

print(
    f"Annotation recordings available: "
    f"{len(annotation_files)}"
)

print(
    "Annotation format: .csv.csv"
)

print(
    "Required columns: "
    + ", ".join(REQUIRED_COLUMNS)
)

print(
    "\n✓ Configuration variables are defined."
)

print(
    "✓ EXPERIMENTS is defined."
)

print(
    "✓ ANO_DIR is defined."
)

print(
    "✓ annotation_files is defined."
)

print(
    "\n✓ READY FOR TRAIN / VALIDATION SPLIT "
    "AND WINDOW GENERATION."
)

print("=" * 80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT INITIALIZATION
Base directory:
  /content/drive/MyDrive/Colab Notebooks/Fetal_project

Annotation directory:
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv

COMPUTING DEVICE
Using device: cpu
CUDA GPU not available. Using CPU.

EXPERIMENT CONFIGURATION
Sampling frequency: 4 Hz
Maximum allowable missingness: 60%
Validation fraction: 20%
Random seed: 42

Required columns:
  ✓ fhr
  ✓ uc
  ✓ fhr_nan
  ✓ uc_nan
  ✓ baseline
  ✓ acc
  ✓ dec

Experiments:

  15min_5min
    Window: 15 min = 3,600 samples
    Stride: 5 min = 1,200 samples

ANNOTATION FILE VERIFICATION
Found .csv.csv annotation files: 134
Valid annotation files: 134
Invalid annotation files: 0

✓ INITIALIZATION COMPLETED
Annotation recordings available: 134
Annotation format: .csv.csv
Required columns: fhr, uc, fhr_nan, uc_nan, baseline, acc, dec

✓ Configuration va

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# RECORDING-LEVEL TRAIN / VALIDATION SPLIT
# Uses ONLY .csv.csv annotation files
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split


# ══════════════════════════════════════════════════════════════════════════════
# 1. FIND THE ANNOTATION FILES
# ══════════════════════════════════════════════════════════════════════════════

all_annotation_paths = sorted(
    glob.glob(
        os.path.join(
            ANO_DIR,
            "*.csv.csv"
        )
    )
)


# Remove duplicates just in case
all_annotation_paths = sorted(
    set(all_annotation_paths)
)


print("=" * 80)
print("ANNOTATION RECORDINGS")
print("=" * 80)

print(
    f"Found .csv.csv files: {len(all_annotation_paths)}"
)


# ══════════════════════════════════════════════════════════════════════════════
# 2. CHECK REQUIRED COLUMNS
# ══════════════════════════════════════════════════════════════════════════════

valid_annotation_paths = []
rejected_annotation_paths = []

for path in all_annotation_paths:

    try:

        df = pd.read_csv(path, nrows=5)

        missing = [
            column
            for column in REQUIRED_COLUMNS
            if column not in df.columns
        ]

        if len(missing) == 0:

            valid_annotation_paths.append(path)

        else:

            rejected_annotation_paths.append(
                (
                    path,
                    f"Missing columns: {missing}"
                )
            )

    except Exception as e:

        rejected_annotation_paths.append(
            (
                path,
                f"READ ERROR: {e}"
            )
        )


print(
    f"Valid annotation files:   "
    f"{len(valid_annotation_paths)}"
)

print(
    f"Rejected annotation files: "
    f"{len(rejected_annotation_paths)}"
)


# ══════════════════════════════════════════════════════════════════════════════
# 3. STOP IF SOMETHING IS WRONG
# ══════════════════════════════════════════════════════════════════════════════

if len(valid_annotation_paths) == 0:

    raise RuntimeError(
        "No valid .csv.csv annotation files were found."
    )


if len(valid_annotation_paths) != 134:

    print()
    print("WARNING:")
    print(
        f"Expected 134 annotation files, "
        f"but found {len(valid_annotation_paths)} valid files."
    )


# ══════════════════════════════════════════════════════════════════════════════
# 4. GET RECORDING NAMES
# ══════════════════════════════════════════════════════════════════════════════

recording_names = sorted(
    [
        os.path.basename(path)
        for path in valid_annotation_paths
    ]
)


# ══════════════════════════════════════════════════════════════════════════════
# 5. RECORDING-LEVEL SPLIT
# ══════════════════════════════════════════════════════════════════════════════

train_names, val_names = train_test_split(

    recording_names,

    test_size=VALIDATION_FRACTION,

    random_state=RANDOM_SEED,

    shuffle=True
)


train_names = sorted(train_names)
val_names = sorted(val_names)


# ══════════════════════════════════════════════════════════════════════════════
# 6. CHECK FOR DATA LEAKAGE
# ══════════════════════════════════════════════════════════════════════════════

train_set = set(train_names)
val_set = set(val_names)

overlap = train_set.intersection(val_set)

if len(overlap) > 0:

    raise RuntimeError(
        "DATA LEAKAGE DETECTED!\n"
        + "\n".join(sorted(overlap))
    )


# Check that every recording was assigned
all_split_names = train_set.union(val_set)

unassigned = set(recording_names) - all_split_names

if len(unassigned) > 0:

    raise RuntimeError(
        "Some recordings were not assigned:\n"
        + "\n".join(sorted(unassigned))
    )


# ══════════════════════════════════════════════════════════════════════════════
# 7. CREATE FULL PATHS
# ══════════════════════════════════════════════════════════════════════════════

train_paths = [
    os.path.join(ANO_DIR, name)
    for name in train_names
]

val_paths = [
    os.path.join(ANO_DIR, name)
    for name in val_names
]


# ══════════════════════════════════════════════════════════════════════════════
# 8. FINAL REPORT
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("FINAL RECORDING-LEVEL SPLIT")
print("=" * 80)

print(
    f"Total recordings:       {len(recording_names)}"
)

print(
    f"Training recordings:    {len(train_names)}"
)

print(
    f"Validation recordings:  {len(val_names)}"
)

print(
    f"Training + validation:  "
    f"{len(train_names) + len(val_names)}"
)

print(
    f"Overlap:                {len(overlap)}"
)

print(
    f"Unassigned:             {len(unassigned)}"
)


print()
print("Training recordings:")

for name in train_names:
    print(f"  {name}")


print()
print("Validation recordings:")

for name in val_names:
    print(f"  {name}")


# ══════════════════════════════════════════════════════════════════════════════
# 9. FINAL SUCCESS CHECK
# ══════════════════════════════════════════════════════════════════════════════

assert len(train_names) + len(val_names) == len(recording_names)
assert len(overlap) == 0
assert len(unassigned) == 0


print()
print("=" * 80)
print("✓ RECORDING-LEVEL SPLIT COMPLETED")
print("=" * 80)

print("✓ Only .csv.csv annotation files were used.")
print("✓ No .csv files were used.")
print("✓ No recording appears in both training and validation.")
print("✓ All recordings are accounted for.")

ANNOTATION RECORDINGS
Found .csv.csv files: 134
Valid annotation files:   134
Rejected annotation files: 0

FINAL RECORDING-LEVEL SPLIT
Total recordings:       134
Training recordings:    107
Validation recordings:  27
Training + validation:  134
Overlap:                0
Unassigned:             0

Training recordings:
  test01.csv.csv
  test02.csv.csv
  test04.csv.csv
  test05.csv.csv
  test07.csv.csv
  test08.csv.csv
  test09.csv.csv
  test11.csv.csv
  test13.csv.csv
  test15.csv.csv
  test17.csv.csv
  test18.csv.csv
  test19.csv.csv
  test22.csv.csv
  test25.csv.csv
  test26.csv.csv
  test29.csv.csv
  test30.csv.csv
  test31.csv.csv
  test32.csv.csv
  test38.csv.csv
  test39.csv.csv
  test40.csv.csv
  test44.csv.csv
  test45.csv.csv
  test46.csv.csv
  test47.csv.csv
  test48.csv.csv
  test52.csv.csv
  test53.csv.csv
  test54.csv.csv
  test56.csv.csv
  test58.csv.csv
  test61.csv.csv
  test64.csv.csv
  test65.csv.csv
  test66.csv.csv
  test67.csv.csv
  test68.csv.csv
  test69.csv.csv

In [4]:
# ============================================================
# VERIFY TRAIN / VALIDATION PATHS
# ============================================================

print("=" * 80)
print("TRAIN / VALIDATION PATH VERIFICATION")
print("=" * 80)

print(f"Training paths:   {len(train_paths)}")
print(f"Validation paths: {len(val_paths)}")

print("\nFirst 5 training paths:")
for p in train_paths[:5]:
    print(" ", p)

print("\nFirst 5 validation paths:")
for p in val_paths[:5]:
    print(" ", p)

# Make absolutely sure no plain .csv files slipped in
assert all(p.endswith(".csv.csv") for p in train_paths)
assert all(p.endswith(".csv.csv") for p in val_paths)

# Make sure all paths exist
assert all(os.path.isfile(p) for p in train_paths)
assert all(os.path.isfile(p) for p in val_paths)

print("\n✓ All training paths are .csv.csv")
print("✓ All validation paths are .csv.csv")
print("✓ All files exist")
print("✓ Ready for Dataset/DataLoader creation")

TRAIN / VALIDATION PATH VERIFICATION
Training paths:   107
Validation paths: 27

First 5 training paths:
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test01.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test02.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test04.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test05.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test07.csv.csv

First 5 validation paths:
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test06.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test14.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test16.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTGDL_FHRMA_ano_csv/test20.csv.csv
  /content/drive/MyDrive/Colab Notebooks/Fetal_project/CTG

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# VERIFY ALL TRAINING / VALIDATION ANNOTATION FILES
# ══════════════════════════════════════════════════════════════════════════════

def verify_annotation_paths(paths, dataset_name):

    print()
    print("=" * 80)
    print(f"VERIFYING {dataset_name.upper()} ANNOTATION FILES")
    print("=" * 80)

    total = len(paths)
    loaded = 0
    failed = 0

    total_rows = 0

    failures = []

    for path in paths:

        filename = os.path.basename(path)

        try:

            df = pd.read_csv(path)

            # Check required columns
            missing = [
                col
                for col in REQUIRED_COLUMNS
                if col not in df.columns
            ]

            if missing:

                failed += 1

                failures.append(
                    (
                        filename,
                        f"Missing columns: {missing}"
                    )
                )

                continue

            # Check that the file contains data
            if len(df) == 0:

                failed += 1

                failures.append(
                    (
                        filename,
                        "File contains zero rows"
                    )
                )

                continue

            loaded += 1
            total_rows += len(df)

        except Exception as e:

            failed += 1

            failures.append(
                (
                    filename,
                    str(e)
                )
            )

    print(f"Files expected:  {total}")
    print(f"Files loaded:    {loaded}")
    print(f"Files failed:    {failed}")
    print(f"Total rows:      {total_rows:,}")

    if failures:

        print()
        print("FAILED FILES:")

        for filename, reason in failures:

            print(
                f"  {filename}: {reason}"
            )

    else:

        print()
        print("✓ Every file loaded successfully.")
        print("✓ Every file contains all required columns.")
        print("✓ Every file contains data.")

    return loaded, failed, total_rows, failures


# ══════════════════════════════════════════════════════════════════════════════
# RUN VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════

train_loaded, train_failed, train_rows, train_failures = \
    verify_annotation_paths(
        train_paths,
        "training"
    )


val_loaded, val_failed, val_rows, val_failures = \
    verify_annotation_paths(
        val_paths,
        "validation"
    )


# ══════════════════════════════════════════════════════════════════════════════
# FINAL CHECK
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("ANNOTATION FILE VERIFICATION SUMMARY")
print("=" * 80)

print(
    f"Training:   {train_loaded}/{len(train_paths)} files loaded"
)

print(
    f"Validation: {val_loaded}/{len(val_paths)} files loaded"
)

print(
    f"Training rows:   {train_rows:,}"
)

print(
    f"Validation rows: {val_rows:,}"
)


if train_failed == 0 and val_failed == 0:

    print()
    print("✓ ALL TRAINING FILES ARE VALID")
    print("✓ ALL VALIDATION FILES ARE VALID")
    print("✓ READY FOR WINDOW GENERATION")

else:

    raise RuntimeError(
        "Some annotation files failed verification. "
        "Do not start training until this is resolved."
    )


VERIFYING TRAINING ANNOTATION FILES
Files expected:  107
Files loaded:    107
Files failed:    0
Total rows:      1,268,485

✓ Every file loaded successfully.
✓ Every file contains all required columns.
✓ Every file contains data.

VERIFYING VALIDATION ANNOTATION FILES
Files expected:  27
Files loaded:    27
Files failed:    0
Total rows:      320,085

✓ Every file loaded successfully.
✓ Every file contains all required columns.
✓ Every file contains data.

ANNOTATION FILE VERIFICATION SUMMARY
Training:   107/107 files loaded
Validation: 27/27 files loaded
Training rows:   1,268,485
Validation rows: 320,085

✓ ALL TRAINING FILES ARE VALID
✓ ALL VALIDATION FILES ARE VALID
✓ READY FOR WINDOW GENERATION


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# WINDOW GENERATION
# Uses ONLY train_paths and val_paths
# Never scans ANO_DIR again
# Never mixes recordings
# ══════════════════════════════════════════════════════════════════════════════

def create_windows_from_paths(
    paths,
    window_size,
    stride_size,
    dataset_name
):

    windows = []

    files_seen = 0
    files_loaded = 0
    files_skipped = 0

    windows_examined = 0
    valid_windows = 0

    skipped_files = []

    print()
    print("=" * 80)
    print(f"CREATING WINDOWS: {dataset_name}")
    print("=" * 80)

    print(
        f"Window size: {window_size} samples"
    )

    print(
        f"Stride:      {stride_size} samples"
    )

    print(
        f"Recordings:  {len(paths)}"
    )

    # ------------------------------------------------------------------
    # PROCESS EACH RECORDING INDEPENDENTLY
    # ------------------------------------------------------------------

    for path in paths:

        files_seen += 1

        filename = os.path.basename(path)

        try:

            df = pd.read_csv(path)

            # ----------------------------------------------------------
            # Required columns
            # ----------------------------------------------------------

            missing = [
                col
                for col in REQUIRED_COLUMNS
                if col not in df.columns
            ]

            if missing:

                files_skipped += 1

                skipped_files.append(
                    (
                        filename,
                        f"Missing columns: {missing}"
                    )
                )

                continue

            files_loaded += 1

            # ----------------------------------------------------------
            # Convert required columns to numeric
            # ----------------------------------------------------------

            data = df[REQUIRED_COLUMNS].apply(
                pd.to_numeric,
                errors="coerce"
            )

            # ----------------------------------------------------------
            # Generate windows INSIDE THIS RECORDING ONLY
            # ----------------------------------------------------------

            n_samples = len(data)

            if n_samples < window_size:

                continue

            for start in range(
                0,
                n_samples - window_size + 1,
                stride_size
            ):

                end = start + window_size

                window = data.iloc[
                    start:end
                ].copy()

                windows_examined += 1

                # ------------------------------------------------------
                # Missingness check
                # ------------------------------------------------------

                fhr_missing_pct = (
                    window["fhr"].isna().mean()
                )

                uc_missing_pct = (
                    window["uc"].isna().mean()
                )

                if (
                    fhr_missing_pct
                    > MAX_ALLOWABLE_NAN_PCT
                ):

                    continue

                if (
                    uc_missing_pct
                    > MAX_ALLOWABLE_NAN_PCT
                ):

                    continue

                # ------------------------------------------------------
                # Store window
                # ------------------------------------------------------

                windows.append(
                    {
                        "recording": filename,
                        "start": start,
                        "end": end,
                        "data": window
                    }
                )

                valid_windows += 1

        except Exception as e:

            files_skipped += 1

            skipped_files.append(
                (
                    filename,
                    f"READ ERROR: {e}"
                )
            )

    # ------------------------------------------------------------------
    # REPORT
    # ------------------------------------------------------------------

    print()
    print("-" * 80)

    print(
        f"Files seen:       {files_seen}"
    )

    print(
        f"Files loaded:     {files_loaded}"
    )

    print(
        f"Files skipped:    {files_skipped}"
    )

    print(
        f"Windows examined: {windows_examined}"
    )

    print(
        f"Valid windows:    {valid_windows}"
    )

    print("-" * 80)

    if skipped_files:

        print()
        print("Skipped files:")

        for filename, reason in skipped_files:

            print(
                f"  {filename}: {reason}"
            )

    return windows


# ══════════════════════════════════════════════════════════════════════════════
# GENERATE WINDOWS FOR BOTH EXPERIMENTS
# ══════════════════════════════════════════════════════════════════════════════

WINDOW_DATA = {}


for experiment_name, config in EXPERIMENTS.items():

    window_size = int(
        config["window_minutes"]
        * 60
        * FS
    )

    stride_size = int(
        config["stride_minutes"]
        * 60
        * FS
    )

    print()
    print("=" * 80)
    print(f"EXPERIMENT: {experiment_name}")
    print("=" * 80)

    train_windows = create_windows_from_paths(
        train_paths,
        window_size,
        stride_size,
        f"{experiment_name} - TRAINING"
    )

    val_windows = create_windows_from_paths(
        val_paths,
        window_size,
        stride_size,
        f"{experiment_name} - VALIDATION"
    )

    WINDOW_DATA[experiment_name] = {
        "train": train_windows,
        "val": val_windows
    }


# ══════════════════════════════════════════════════════════════════════════════
# FINAL WINDOW SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("FINAL WINDOW DATASET SUMMARY")
print("=" * 80)

for experiment_name in EXPERIMENTS:

    train_count = len(
        WINDOW_DATA[experiment_name]["train"]
    )

    val_count = len(
        WINDOW_DATA[experiment_name]["val"]
    )

    print()
    print(experiment_name)

    print(
        f"  Training windows:   {train_count}"
    )

    print(
        f"  Validation windows: {val_count}"
    )

print()
print("✓ Window generation completed.")


EXPERIMENT: 15min_5min

CREATING WINDOWS: 15min_5min - TRAINING
Window size: 3600 samples
Stride:      1200 samples
Recordings:  107

--------------------------------------------------------------------------------
Files seen:       107
Files loaded:     107
Files skipped:    0
Windows examined: 749
Valid windows:    749
--------------------------------------------------------------------------------

CREATING WINDOWS: 15min_5min - VALIDATION
Window size: 3600 samples
Stride:      1200 samples
Recordings:  27

--------------------------------------------------------------------------------
Files seen:       27
Files loaded:     27
Files skipped:    0
Windows examined: 189
Valid windows:    189
--------------------------------------------------------------------------------

FINAL WINDOW DATASET SUMMARY

15min_5min
  Training windows:   749
  Validation windows: 189

✓ Window generation completed.


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# PYTORCH DATASET + DATALOADER CREATION
#
# Uses ONLY the already-generated WINDOW_DATA
# Does NOT scan ANO_DIR
# Does NOT reload recordings
# Does NOT change the train/validation split
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# DATASET CLASS
# ══════════════════════════════════════════════════════════════════════════════

class CTGWindowDataset(Dataset):

    def __init__(self, windows):

        self.windows = windows

    def __len__(self):

        return len(self.windows)

    def __getitem__(self, idx):

        item = self.windows[idx]

        data = item["data"]

        # --------------------------------------------------------------
        # Input signals
        # --------------------------------------------------------------

        fhr = data["fhr"].to_numpy(dtype=np.float32)

        uc = data["uc"].to_numpy(dtype=np.float32)

        # --------------------------------------------------------------
        # Annotation targets
        # --------------------------------------------------------------

        baseline = data["baseline"].to_numpy(dtype=np.float32)

        acc = data["acc"].to_numpy(dtype=np.float32)

        dec = data["dec"].to_numpy(dtype=np.float32)

        # --------------------------------------------------------------
        # Missingness indicators
        # --------------------------------------------------------------

        fhr_nan = data["fhr_nan"].to_numpy(dtype=np.float32)

        uc_nan = data["uc_nan"].to_numpy(dtype=np.float32)

        # --------------------------------------------------------------
        # Replace remaining NaN/inf values in input signals
        #
        # The window-level missingness filter has already been applied.
        # This prevents NaN values from entering the neural network.
        # --------------------------------------------------------------

        fhr = np.nan_to_num(
            fhr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        uc = np.nan_to_num(
            uc,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        fhr_nan = np.nan_to_num(
            fhr_nan,
            nan=1.0,
            posinf=1.0,
            neginf=1.0
        )

        uc_nan = np.nan_to_num(
            uc_nan,
            nan=1.0,
            posinf=1.0,
            neginf=1.0
        )

        baseline = np.nan_to_num(
            baseline,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        acc = np.nan_to_num(
            acc,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        dec = np.nan_to_num(
            dec,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        # --------------------------------------------------------------
        # Input tensor
        #
        # Shape:
        #   [channels, samples]
        #
        # Channels:
        #   0 = FHR
        #   1 = UC
        #   2 = FHR missingness
        #   3 = UC missingness
        # --------------------------------------------------------------

        x = np.stack(
            [
                fhr,
                uc,
                fhr_nan,
                uc_nan
            ],
            axis=0
        )

        # --------------------------------------------------------------
        # Target tensor
        #
        # Shape:
        #   [3, samples]
        #
        # Channels:
        #   0 = baseline
        #   1 = acceleration
        #   2 = deceleration
        # --------------------------------------------------------------

        y = np.stack(
            [
                baseline,
                acc,
                dec
            ],
            axis=0
        )

        return (
            torch.from_numpy(x),
            torch.from_numpy(y)
        )


# ══════════════════════════════════════════════════════════════════════════════
# CREATE DATASETS AND DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

BATCH_SIZE = 8

NUM_WORKERS = 0


DATASETS = {}

DATALOADERS = {}


print("=" * 80)
print("CREATING PYTORCH DATASETS")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_windows = WINDOW_DATA[experiment_name]["train"]

    val_windows = WINDOW_DATA[experiment_name]["val"]

    train_dataset = CTGWindowDataset(
        train_windows
    )

    val_dataset = CTGWindowDataset(
        val_windows
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    DATASETS[experiment_name] = {
        "train": train_dataset,
        "val": val_dataset
    }

    DATALOADERS[experiment_name] = {
        "train": train_loader,
        "val": val_loader
    }


# ══════════════════════════════════════════════════════════════════════════════
# VERIFY DATASET / DATALOADER CREATION
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("DATASET / DATALOADER SUMMARY")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_dataset = DATASETS[experiment_name]["train"]

    val_dataset = DATASETS[experiment_name]["val"]

    train_loader = DATALOADERS[experiment_name]["train"]

    val_loader = DATALOADERS[experiment_name]["val"]

    print()
    print(experiment_name)

    print(
        f"  Training samples:   {len(train_dataset):,}"
    )

    print(
        f"  Validation samples: {len(val_dataset):,}"
    )

    print(
        f"  Training batches:   {len(train_loader):,}"
    )

    print(
        f"  Validation batches: {len(val_loader):,}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# TEST ONE BATCH FROM EACH EXPERIMENT
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("TESTING DATALOADERS")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_loader = DATALOADERS[experiment_name]["train"]

    val_loader = DATALOADERS[experiment_name]["val"]

    train_x, train_y = next(iter(train_loader))

    val_x, val_y = next(iter(val_loader))

    print()
    print(experiment_name)

    print(
        f"  Training X shape: {tuple(train_x.shape)}"
    )

    print(
        f"  Training Y shape: {tuple(train_y.shape)}"
    )

    print(
        f"  Validation X shape: {tuple(val_x.shape)}"
    )

    print(
        f"  Validation Y shape: {tuple(val_y.shape)}"
    )

    print(
        f"  X dtype: {train_x.dtype}"
    )

    print(
        f"  Y dtype: {train_y.dtype}"
    )

    print(
        f"  X finite: {torch.isfinite(train_x).all().item()}"
    )

    print(
        f"  Y finite: {torch.isfinite(train_y).all().item()}"
    )


print()
print("=" * 80)
print("✓ PYTORCH DATASETS READY")
print("=" * 80)

CREATING PYTORCH DATASETS

DATASET / DATALOADER SUMMARY

15min_5min
  Training samples:   749
  Validation samples: 189
  Training batches:   94
  Validation batches: 24

TESTING DATALOADERS

15min_5min
  Training X shape: (8, 4, 3600)
  Training Y shape: (8, 3, 3600)
  Validation X shape: (8, 4, 3600)
  Validation Y shape: (8, 3, 3600)
  X dtype: torch.float32
  Y dtype: torch.float32
  X finite: True
  Y finite: True

✓ PYTORCH DATASETS READY


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL DEFINITION
#
# Input:
#     [batch, 4, samples]
#
#     4 channels:
#       0 = FHR
#       1 = UC
#       2 = FHR missingness
#       3 = UC missingness
#
# Output:
#     [batch, 3, samples]
#
#     3 channels:
#       0 = baseline
#       1 = acceleration
#       2 = deceleration
#
# The model preserves the temporal length of the input.
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONVOLUTIONAL BLOCK
# ══════════════════════════════════════════════════════════════════════════════

class ConvBlock1D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv1d(
                out_channels,
                out_channels,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


# ══════════════════════════════════════════════════════════════════════════════
# 2. 1D U-NET
# ══════════════════════════════════════════════════════════════════════════════

class CTGUNet1D(nn.Module):

    def __init__(
        self,
        in_channels=4,
        out_channels=3
    ):

        super().__init__()


        # --------------------------------------------------------------
        # Encoder
        # --------------------------------------------------------------

        self.enc1 = ConvBlock1D(
            in_channels,
            32
        )

        self.pool1 = nn.MaxPool1d(
            kernel_size=2,
            stride=2
        )


        self.enc2 = ConvBlock1D(
            32,
            64
        )

        self.pool2 = nn.MaxPool1d(
            kernel_size=2,
            stride=2
        )


        self.enc3 = ConvBlock1D(
            64,
            128
        )

        self.pool3 = nn.MaxPool1d(
            kernel_size=2,
            stride=2
        )


        # --------------------------------------------------------------
        # Bottleneck
        # --------------------------------------------------------------

        self.bottleneck = ConvBlock1D(
            128,
            256
        )


        # --------------------------------------------------------------
        # Decoder
        # --------------------------------------------------------------

        self.up3 = nn.ConvTranspose1d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = ConvBlock1D(
            256,
            128
        )


        self.up2 = nn.ConvTranspose1d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = ConvBlock1D(
            128,
            64
        )


        self.up1 = nn.ConvTranspose1d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = ConvBlock1D(
            64,
            32
        )


        # --------------------------------------------------------------
        # Output layer
        # --------------------------------------------------------------

        self.output = nn.Conv1d(
            32,
            out_channels,
            kernel_size=1
        )


    def forward(self, x):

        # --------------------------------------------------------------
        # Encoder
        # --------------------------------------------------------------

        e1 = self.enc1(x)

        p1 = self.pool1(e1)


        e2 = self.enc2(p1)

        p2 = self.pool2(e2)


        e3 = self.enc3(p2)

        p3 = self.pool3(e3)


        # --------------------------------------------------------------
        # Bottleneck
        # --------------------------------------------------------------

        b = self.bottleneck(p3)


        # --------------------------------------------------------------
        # Decoder
        # --------------------------------------------------------------

        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(d3)


        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(d2)


        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(d1)


        # --------------------------------------------------------------
        # Output
        # --------------------------------------------------------------

        out = self.output(d1)

        return out


# ══════════════════════════════════════════════════════════════════════════════
# 3. CREATE MODEL
# ══════════════════════════════════════════════════════════════════════════════

model = CTGUNet1D(
    in_channels=4,
    out_channels=3
)


model = model.to(device)


# ══════════════════════════════════════════════════════════════════════════════
# 4. MODEL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("MODEL INITIALIZATION")
print("=" * 80)

print(
    f"Model: {model.__class__.__name__}"
)

print(
    f"Input channels:  4"
)

print(
    f"Output channels: 3"
)

print(
    f"Device: {device}"
)


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total parameters:     {total_parameters:,}"
)

print(
    f"Trainable parameters:  {trainable_parameters:,}"
)


# ══════════════════════════════════════════════════════════════════════════════
# 5. TEST MODEL WITH BOTH WINDOW SIZES
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("MODEL SHAPE VERIFICATION")
print("=" * 80)


model.eval()


with torch.no_grad():

    for experiment_name in EXPERIMENTS:

        train_loader = (
            DATALOADERS[
                experiment_name
            ]["train"]
        )

        x, y = next(
            iter(train_loader)
        )

        x = x.to(device)
        y = y.to(device)

        prediction = model(x)

        print()
        print(experiment_name)

        print(
            f"  Input shape:      {tuple(x.shape)}"
        )

        print(
            f"  Target shape:     {tuple(y.shape)}"
        )

        print(
            f"  Prediction shape: {tuple(prediction.shape)}"
        )

        print(
            f"  Input finite:      "
            f"{torch.isfinite(x).all().item()}"
        )

        print(
            f"  Target finite:     "
            f"{torch.isfinite(y).all().item()}"
        )

        print(
            f"  Prediction finite: "
            f"{torch.isfinite(prediction).all().item()}"
        )

        assert prediction.shape == y.shape, (
            f"Shape mismatch for {experiment_name}: "
            f"prediction={prediction.shape}, "
            f"target={y.shape}"
        )


# ══════════════════════════════════════════════════════════════════════════════
# 6. FINAL MODEL CHECK
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("✓ MODEL VERIFICATION COMPLETED")
print("=" * 80)

print(
    "✓ Model accepts 4-channel CTG input."
)

print(
    "✓ Model produces 3-channel annotation output."
)

print(
    "✓ X-minute window shape verified."
)

#print(
#    "✓ 7.5-minute window shape verified."
#)

print(
    "✓ Prediction contains only finite values."
)

print(
    "✓ Model is ready for loss-function and training setup."
)

MODEL INITIALIZATION
Model: CTGUNet1D
Input channels:  4
Output channels: 3
Device: cpu
Total parameters:     1,062,595
Trainable parameters:  1,062,595

MODEL SHAPE VERIFICATION

15min_5min
  Input shape:      (8, 4, 3600)
  Target shape:     (8, 3, 3600)
  Prediction shape: (8, 3, 3600)
  Input finite:      True
  Target finite:     True
  Prediction finite: True

✓ MODEL VERIFICATION COMPLETED
✓ Model accepts 4-channel CTG input.
✓ Model produces 3-channel annotation output.
✓ X-minute window shape verified.
✓ Prediction contains only finite values.
✓ Model is ready for loss-function and training setup.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSS FUNCTION + OPTIMIZER — CLASS-WEIGHTED VERSION
#
# Output channels:
#   0 = baseline
#   1 = acceleration
#   2 = deceleration
#
# ACC and DEC use positive-class weighting to handle class imbalance.
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# 1. LOSS FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

class CTGMultiTaskLoss(nn.Module):

    def __init__(
        self,
        baseline_weight=1.0,
        acc_weight=1.0,
        dec_weight=1.0,
        acc_pos_weight=10.0,
        dec_pos_weight=1.5
    ):

        super().__init__()

        self.baseline_weight = baseline_weight
        self.acc_weight = acc_weight
        self.dec_weight = dec_weight

        # --------------------------------------------------------------
        # Baseline
        # --------------------------------------------------------------

        self.baseline_loss = nn.SmoothL1Loss()

        # --------------------------------------------------------------
        # ACC / DEC
        #
        # pos_weight increases the penalty for missing positive events.
        #
        # ACC is highly imbalanced, so it receives a larger weight.
        # DEC is less imbalanced, so its weight is smaller.
        # --------------------------------------------------------------

        self.acc_loss = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                acc_pos_weight,
                dtype=torch.float32,
                device=device
            )
        )

        self.dec_loss = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                dec_pos_weight,
                dtype=torch.float32,
                device=device
            )
        )


    def forward(self, prediction, target):

        # --------------------------------------------------------------
        # Separate output channels
        # --------------------------------------------------------------

        baseline_pred = prediction[:, 0, :]
        acc_pred = prediction[:, 1, :]
        dec_pred = prediction[:, 2, :]

        baseline_target = target[:, 0, :]
        acc_target = target[:, 1, :]
        dec_target = target[:, 2, :]


        # --------------------------------------------------------------
        # Individual losses
        # --------------------------------------------------------------

        baseline_loss = self.baseline_loss(
            baseline_pred,
            baseline_target
        )

        acc_loss = self.acc_loss(
            acc_pred,
            acc_target
        )

        dec_loss = self.dec_loss(
            dec_pred,
            dec_target
        )


        # --------------------------------------------------------------
        # Combined loss
        # --------------------------------------------------------------

        total_loss = (

            self.baseline_weight * baseline_loss

            + self.acc_weight * acc_loss

            + self.dec_weight * dec_loss

        )


        return total_loss, {

            "baseline": baseline_loss.detach(),

            "acc": acc_loss.detach(),

            "dec": dec_loss.detach()

        }


# ══════════════════════════════════════════════════════════════════════════════
# 2. LOSS WEIGHTS
# ══════════════════════════════════════════════════════════════════════════════

ACC_POS_WEIGHT = 10.0

DEC_POS_WEIGHT = 1.5


criterion = CTGMultiTaskLoss(

    baseline_weight=1.0,

    acc_weight=1.0,

    dec_weight=1.0,

    acc_pos_weight=ACC_POS_WEIGHT,

    dec_pos_weight=DEC_POS_WEIGHT
)


# ══════════════════════════════════════════════════════════════════════════════
# 3. OPTIMIZER
# ══════════════════════════════════════════════════════════════════════════════

LEARNING_RATE = 3e-4 #1e-3

WEIGHT_DECAY = 1e-5


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)


# ══════════════════════════════════════════════════════════════════════════════
# 4. LEARNING-RATE SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=3

)


# ══════════════════════════════════════════════════════════════════════════════
# 5. VERIFY LOSS ON ONE BATCH
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("CLASS-WEIGHTED LOSS + OPTIMIZER VERIFICATION")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_loader = DATALOADERS[
        experiment_name
    ]["train"]


    x, y = next(
        iter(train_loader)
    )


    x = x.to(device)
    y = y.to(device)


    model.train()

    optimizer.zero_grad()


    prediction = model(x)


    loss, loss_components = criterion(

        prediction,

        y

    )


    print()
    print(experiment_name)

    print(
        f"  Total loss:      {loss.item():.6f}"
    )

    print(
        f"  Baseline loss:   "
        f"{loss_components['baseline'].item():.6f}"
    )

    print(
        f"  Acc loss:        "
        f"{loss_components['acc'].item():.6f}"
    )

    print(
        f"  Dec loss:        "
        f"{loss_components['dec'].item():.6f}"
    )

    print(
        f"  ACC pos weight:  "
        f"{ACC_POS_WEIGHT}"
    )

    print(
        f"  DEC pos weight:  "
        f"{DEC_POS_WEIGHT}"
    )

    print(
        f"  Learning rate:   "
        f"{optimizer.param_groups[0]['lr']:.6f}"
    )


    assert torch.isfinite(loss).item(), (

        f"Non-finite loss detected for "

        f"{experiment_name}"

    )


# ══════════════════════════════════════════════════════════════════════════════
# 6. FINAL VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("✓ CLASS-WEIGHTED LOSS READY")
print("=" * 80)

print(
    f"Learning rate:     {LEARNING_RATE}"
)

print(
    f"Weight decay:      {WEIGHT_DECAY}"
)

print(
    "Baseline loss:     SmoothL1"
)

print(
    "ACC loss:          BCEWithLogitsLoss"
)

print(
    "DEC loss:          BCEWithLogitsLoss"
)

print(
    f"ACC pos_weight:    {ACC_POS_WEIGHT}"
)

print(
    f"DEC pos_weight:    {DEC_POS_WEIGHT}"
)

print(
    "✓ Loss is finite."
)

print(
    "✓ Optimizer initialized."
)

print(
    "✓ Scheduler initialized."
)

print(
    "✓ Class imbalance correction enabled."
)

print(
    "✓ Ready for training loop."
)

CLASS-WEIGHTED LOSS + OPTIMIZER VERIFICATION

15min_5min
  Total loss:      304.104279
  Baseline loss:   169.901047
  Acc loss:        78.823135
  Dec loss:        55.380108
  ACC pos weight:  10.0
  DEC pos weight:  1.5
  Learning rate:   0.000300

✓ CLASS-WEIGHTED LOSS READY
Learning rate:     0.0003
Weight decay:      1e-05
Baseline loss:     SmoothL1
ACC loss:          BCEWithLogitsLoss
DEC loss:          BCEWithLogitsLoss
ACC pos_weight:    10.0
DEC pos_weight:    1.5
✓ Loss is finite.
✓ Optimizer initialized.
✓ Scheduler initialized.
✓ Class imbalance correction enabled.
✓ Ready for training loop.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# VERIFY TARGET ENCODING BEFORE TRAINING
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("VERIFYING TARGET ENCODING")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_dataset = DATASETS[
        experiment_name
    ]["train"]

    val_dataset = DATASETS[
        experiment_name
    ]["val"]


    print()
    print("-" * 80)
    print(experiment_name)
    print("-" * 80)


    # ------------------------------------------------------------------
    # Collect statistics from training windows
    # ------------------------------------------------------------------

    baseline_values = []
    acc_values = []
    dec_values = []


    # We inspect every window, but only the target channels.
    for item in train_dataset.windows:

        data = item["data"]


        baseline = pd.to_numeric(
            data["baseline"],
            errors="coerce"
        ).to_numpy(
            dtype=np.float32
        )

        acc = pd.to_numeric(
            data["acc"],
            errors="coerce"
        ).to_numpy(
            dtype=np.float32
        )

        dec = pd.to_numeric(
            data["dec"],
            errors="coerce"
        ).to_numpy(
            dtype=np.float32
        )


        baseline_values.append(
            baseline
        )

        acc_values.append(
            acc
        )

        dec_values.append(
            dec
        )


    baseline_values = np.concatenate(
        baseline_values
    )

    acc_values = np.concatenate(
        acc_values
    )

    dec_values = np.concatenate(
        dec_values
    )


    # ------------------------------------------------------------------
    # Remove NaN/Inf for statistics
    # ------------------------------------------------------------------

    baseline_finite = baseline_values[
        np.isfinite(baseline_values)
    ]

    acc_finite = acc_values[
        np.isfinite(acc_values)
    ]

    dec_finite = dec_values[
        np.isfinite(dec_values)
    ]


    # ------------------------------------------------------------------
    # Baseline statistics
    # ------------------------------------------------------------------

    print()
    print("BASELINE")

    print(
        f"  Finite values: {len(baseline_finite):,}"
    )

    if len(baseline_finite) > 0:

        print(
            f"  Min:    {np.min(baseline_finite):.4f}"
        )

        print(
            f"  Max:    {np.max(baseline_finite):.4f}"
        )

        print(
            f"  Mean:   {np.mean(baseline_finite):.4f}"
        )

        print(
            f"  Median: {np.median(baseline_finite):.4f}"
        )


    # ------------------------------------------------------------------
    # Acceleration statistics
    # ------------------------------------------------------------------

    print()
    print("ACCELERATION")

    print(
        f"  Finite values: {len(acc_finite):,}"
    )

    if len(acc_finite) > 0:

        unique_acc = np.unique(
            acc_finite
        )

        print(
            f"  Min: {np.min(acc_finite):.4f}"
        )

        print(
            f"  Max: {np.max(acc_finite):.4f}"
        )

        print(
            f"  Mean: {np.mean(acc_finite):.6f}"
        )

        print(
            f"  Unique values: "
            f"{unique_acc[:30]}"
        )

        if np.all(
            np.isin(
                unique_acc,
                [0.0, 1.0]
            )
        ):

            print(
                "  ✓ ACC is binary 0/1."
            )

        else:

            print(
                "  ⚠ ACC is NOT binary 0/1."
            )


    # ------------------------------------------------------------------
    # Deceleration statistics
    # ------------------------------------------------------------------

    print()
    print("DECELERATION")

    print(
        f"  Finite values: {len(dec_finite):,}"
    )

    if len(dec_finite) > 0:

        unique_dec = np.unique(
            dec_finite
        )

        print(
            f"  Min: {np.min(dec_finite):.4f}"
        )

        print(
            f"  Max: {np.max(dec_finite):.4f}"
        )

        print(
            f"  Mean: {np.mean(dec_finite):.6f}"
        )

        print(
            f"  Unique values: "
            f"{unique_dec[:30]}"
        )

        if np.all(
            np.isin(
                unique_dec,
                [0.0, 1.0]
            )
        ):

            print(
                "  ✓ DEC is binary 0/1."
            )

        else:

            print(
                "  ⚠ DEC is NOT binary 0/1."
            )


print()
print("=" * 80)
print("✓ TARGET ENCODING CHECK COMPLETED")
print("=" * 80)

VERIFYING TARGET ENCODING

--------------------------------------------------------------------------------
15min_5min
--------------------------------------------------------------------------------

BASELINE
  Finite values: 2,696,400
  Min:    162.9432
  Max:    177.9872
  Mean:   170.8226
  Median: 170.9662

ACCELERATION
  Finite values: 2,696,400
  Min: 0.0000
  Max: 200.0000
  Mean: 7.261905
  Unique values: [  0. 200.]
  ⚠ ACC is NOT binary 0/1.

DECELERATION
  Finite values: 2,696,400
  Min: 0.0000
  Max: 200.0000
  Mean: 88.166618
  Unique values: [  0. 200.]
  ⚠ DEC is NOT binary 0/1.

✓ TARGET ENCODING CHECK COMPLETED


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# NORMALIZE ACCELERATION / DECELERATION TARGETS
#
# Original annotation encoding:
#     ACC: 0 or 200
#     DEC: 0 or 200
#
# Required encoding for BCEWithLogitsLoss:
#     ACC: 0 or 1
#     DEC: 0 or 1
#
# This block does NOT modify the original CSV files.
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# 1. UPDATE DATASET CLASS
# ══════════════════════════════════════════════════════════════════════════════

class CTGWindowDataset(Dataset):

    def __init__(self, windows):

        self.windows = windows


    def __len__(self):

        return len(self.windows)


    def __getitem__(self, idx):

        item = self.windows[idx]

        data = item["data"]


        # ------------------------------------------------------------------
        # INPUT SIGNALS
        # ------------------------------------------------------------------

        fhr = data["fhr"].to_numpy(
            dtype=np.float32
        )

        uc = data["uc"].to_numpy(
            dtype=np.float32
        )


        # ------------------------------------------------------------------
        # TARGETS
        # ------------------------------------------------------------------

        baseline = data["baseline"].to_numpy(
            dtype=np.float32
        )

        acc = data["acc"].to_numpy(
            dtype=np.float32
        )

        dec = data["dec"].to_numpy(
            dtype=np.float32
        )


        # ------------------------------------------------------------------
        # MISSINGNESS INDICATORS
        # ------------------------------------------------------------------

        fhr_nan = data["fhr_nan"].to_numpy(
            dtype=np.float32
        )

        uc_nan = data["uc_nan"].to_numpy(
            dtype=np.float32
        )


        # ------------------------------------------------------------------
        # REPLACE INVALID INPUT VALUES
        # ------------------------------------------------------------------

        fhr = np.nan_to_num(
            fhr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        uc = np.nan_to_num(
            uc,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        fhr_nan = np.nan_to_num(
            fhr_nan,
            nan=1.0,
            posinf=1.0,
            neginf=1.0
        )

        uc_nan = np.nan_to_num(
            uc_nan,
            nan=1.0,
            posinf=1.0,
            neginf=1.0
        )

        baseline = np.nan_to_num(
            baseline,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        acc = np.nan_to_num(
            acc,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        dec = np.nan_to_num(
            dec,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )


        # ------------------------------------------------------------------
        # NORMALIZE EVENT TARGETS
        #
        # Original:
        #     0   = no event
        #     200 = event
        #
        # New:
        #     0 = no event
        #     1 = event
        # ------------------------------------------------------------------

        acc = acc / 200.0

        dec = dec / 200.0


        # ------------------------------------------------------------------
        # INPUT TENSOR
        #
        # [4, samples]
        #
        #   0 = FHR
        #   1 = UC
        #   2 = FHR missingness
        #   3 = UC missingness
        # ------------------------------------------------------------------

        x = np.stack(
            [
                fhr,
                uc,
                fhr_nan,
                uc_nan
            ],
            axis=0
        )


        # ------------------------------------------------------------------
        # TARGET TENSOR
        #
        # [3, samples]
        #
        #   0 = baseline
        #   1 = acceleration, now 0/1
        #   2 = deceleration, now 0/1
        # ------------------------------------------------------------------

        y = np.stack(
            [
                baseline,
                acc,
                dec
            ],
            axis=0
        )


        return (
            torch.from_numpy(x),
            torch.from_numpy(y)
        )


# ══════════════════════════════════════════════════════════════════════════════
# 2. RECREATE DATASETS / DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

BATCH_SIZE = 8

NUM_WORKERS = 0


DATASETS = {}

DATALOADERS = {}


print("=" * 80)
print("REBUILDING PYTORCH DATASETS WITH NORMALIZED TARGETS")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_windows = WINDOW_DATA[
        experiment_name
    ]["train"]

    val_windows = WINDOW_DATA[
        experiment_name
    ]["val"]


    train_dataset = CTGWindowDataset(
        train_windows
    )

    val_dataset = CTGWindowDataset(
        val_windows
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )


    DATASETS[experiment_name] = {
        "train": train_dataset,
        "val": val_dataset
    }

    DATALOADERS[experiment_name] = {
        "train": train_loader,
        "val": val_loader
    }


# ══════════════════════════════════════════════════════════════════════════════
# 3. VERIFY NORMALIZED TARGETS
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("VERIFYING NORMALIZED TARGET ENCODING")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    train_dataset = DATASETS[
        experiment_name
    ]["train"]

    val_dataset = DATASETS[
        experiment_name
    ]["val"]


    # --------------------------------------------------------------
    # Inspect first training sample
    # --------------------------------------------------------------

    train_x, train_y = train_dataset[0]

    val_x, val_y = val_dataset[0]


    acc_train = train_y[1].numpy()
    dec_train = train_y[2].numpy()

    acc_val = val_y[1].numpy()
    dec_val = val_y[2].numpy()


    print()
    print("-" * 80)
    print(experiment_name)
    print("-" * 80)


    print(
        f"  Training X shape: {tuple(train_x.shape)}"
    )

    print(
        f"  Training Y shape: {tuple(train_y.shape)}"
    )

    print(
        f"  Validation X shape: {tuple(val_x.shape)}"
    )

    print(
        f"  Validation Y shape: {tuple(val_y.shape)}"
    )


    print()
    print("  Training ACC unique values:")

    print(
        f"    {np.unique(acc_train)}"
    )


    print("  Training DEC unique values:")

    print(
        f"    {np.unique(dec_train)}"
    )


    print()
    print("  Validation ACC unique values:")

    print(
        f"    {np.unique(acc_val)}"
    )


    print("  Validation DEC unique values:")

    print(
        f"    {np.unique(dec_val)}"
    )


    # --------------------------------------------------------------
    # Verify binary encoding
    # --------------------------------------------------------------

    assert np.all(
        np.isin(
            np.unique(acc_train),
            [0.0, 1.0]
        )
    ), (
        f"Training ACC is not binary for "
        f"{experiment_name}"
    )


    assert np.all(
        np.isin(
            np.unique(dec_train),
            [0.0, 1.0]
        )
    ), (
        f"Training DEC is not binary for "
        f"{experiment_name}"
    )


    assert np.all(
        np.isin(
            np.unique(acc_val),
            [0.0, 1.0]
        )
    ), (
        f"Validation ACC is not binary for "
        f"{experiment_name}"
    )


    assert np.all(
        np.isin(
            np.unique(dec_val),
            [0.0, 1.0]
        )
    ), (
        f"Validation DEC is not binary for "
        f"{experiment_name}"
    )


    # --------------------------------------------------------------
    # Verify target range
    # --------------------------------------------------------------

    assert np.min(acc_train) >= 0.0
    assert np.max(acc_train) <= 1.0

    assert np.min(dec_train) >= 0.0
    assert np.max(dec_train) <= 1.0

    assert np.min(acc_val) >= 0.0
    assert np.max(acc_val) <= 1.0

    assert np.min(dec_val) >= 0.0
    assert np.max(dec_val) <= 1.0


print()
print("=" * 80)
print("✓ TARGET NORMALIZATION VERIFIED")
print("=" * 80)

print("✓ ACC converted from 0/200 to 0/1.")
print("✓ DEC converted from 0/200 to 0/1.")
print("✓ Original CSV annotation files were not modified.")
print("✓ ACC is compatible with BCEWithLogitsLoss.")
print("✓ DEC is compatible with BCEWithLogitsLoss.")

REBUILDING PYTORCH DATASETS WITH NORMALIZED TARGETS

VERIFYING NORMALIZED TARGET ENCODING

--------------------------------------------------------------------------------
15min_5min
--------------------------------------------------------------------------------
  Training X shape: (4, 3600)
  Training Y shape: (3, 3600)
  Validation X shape: (4, 3600)
  Validation Y shape: (3, 3600)

  Training ACC unique values:
    [0. 1.]
  Training DEC unique values:
    [0. 1.]

  Validation ACC unique values:
    [0. 1.]
  Validation DEC unique values:
    [0. 1.]

✓ TARGET NORMALIZATION VERIFIED
✓ ACC converted from 0/200 to 0/1.
✓ DEC converted from 0/200 to 0/1.
✓ Original CSV annotation files were not modified.
✓ ACC is compatible with BCEWithLogitsLoss.
✓ DEC is compatible with BCEWithLogitsLoss.


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# RE-VERIFY LOSS AFTER TARGET NORMALIZATION
#
# ACC / DEC are now encoded as 0/1.
# This block confirms BCEWithLogitsLoss is receiving valid targets.
# ══════════════════════════════════════════════════════════════════════════════


print("=" * 80)
print("RE-VERIFYING LOSS AFTER TARGET NORMALIZATION")
print("=" * 80)


model.train()


for experiment_name in EXPERIMENTS:

    train_loader = DATALOADERS[
        experiment_name
    ]["train"]


    # --------------------------------------------------------------
    # Get one batch
    # --------------------------------------------------------------

    x, y = next(
        iter(train_loader)
    )


    x = x.to(device)
    y = y.to(device)


    # --------------------------------------------------------------
    # Verify target values before loss
    # --------------------------------------------------------------

    baseline_target = y[:, 0, :]
    acc_target = y[:, 1, :]
    dec_target = y[:, 2, :]


    print()
    print("-" * 80)
    print(experiment_name)
    print("-" * 80)


    print(
        f"Input shape:  {tuple(x.shape)}"
    )

    print(
        f"Target shape: {tuple(y.shape)}"
    )


    print()
    print("Target ranges:")

    print(
        f"  Baseline: "
        f"{baseline_target.min().item():.4f} "
        f"to "
        f"{baseline_target.max().item():.4f}"
    )

    print(
        f"  ACC:      "
        f"{acc_target.min().item():.4f} "
        f"to "
        f"{acc_target.max().item():.4f}"
    )

    print(
        f"  DEC:      "
        f"{dec_target.min().item():.4f} "
        f"to "
        f"{dec_target.max().item():.4f}"
    )


    # --------------------------------------------------------------
    # Verify binary targets
    # --------------------------------------------------------------

    acc_unique = torch.unique(
        acc_target
    )

    dec_unique = torch.unique(
        dec_target
    )


    print()
    print(
        f"  ACC unique values: {acc_unique.cpu().numpy()}"
    )

    print(
        f"  DEC unique values: {dec_unique.cpu().numpy()}"
    )


    assert torch.all(
        (acc_target == 0) |
        (acc_target == 1)
    ), (
        f"Invalid ACC target values "
        f"for {experiment_name}"
    )


    assert torch.all(
        (dec_target == 0) |
        (dec_target == 1)
    ), (
        f"Invalid DEC target values "
        f"for {experiment_name}"
    )


    # --------------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------------

    optimizer.zero_grad()

    prediction = model(x)


    # --------------------------------------------------------------
    # Calculate loss
    # --------------------------------------------------------------

    loss, loss_components = criterion(
        prediction,
        y
    )


    # --------------------------------------------------------------
    # Verify loss values
    # --------------------------------------------------------------

    print()
    print("Loss:")

    print(
        f"  Total:    {loss.item():.6f}"
    )

    print(
        f"  Baseline: "
        f"{loss_components['baseline'].item():.6f}"
    )

    print(
        f"  ACC:      "
        f"{loss_components['acc'].item():.6f}"
    )

    print(
        f"  DEC:      "
        f"{loss_components['dec'].item():.6f}"
    )


    assert torch.isfinite(loss).item(), (
        f"Non-finite total loss for "
        f"{experiment_name}"
    )


    assert torch.isfinite(
        loss_components["baseline"]
    ).item()


    assert torch.isfinite(
        loss_components["acc"]
    ).item()


    assert torch.isfinite(
        loss_components["dec"]
    ).item()


    # --------------------------------------------------------------
    # Test backward pass
    # --------------------------------------------------------------

    loss.backward()


    # Check that gradients are finite
    # --------------------------------------------------------------

    nonfinite_gradients = []

    for name, parameter in model.named_parameters():

        if parameter.grad is not None:

            if not torch.isfinite(
                parameter.grad
            ).all():

                nonfinite_gradients.append(
                    name
                )


    assert len(nonfinite_gradients) == 0, (
        "Non-finite gradients detected:\n"
        + "\n".join(nonfinite_gradients)
    )


    # Clear gradients after verification
    optimizer.zero_grad()


    print()
    print("✓ Forward pass successful.")
    print("✓ Loss is finite.")
    print("✓ Backward pass successful.")
    print("✓ Gradients are finite.")


print()
print("=" * 80)
print("✓ LOSS VERIFICATION COMPLETED")
print("=" * 80)

print("✓ Baseline uses SmoothL1Loss.")
print("✓ ACC uses BCEWithLogitsLoss with 0/1 targets.")
print("✓ DEC uses BCEWithLogitsLoss with 0/1 targets.")
print("✓ Forward pass is valid.")
print("✓ Backward pass is valid.")
print("✓ Gradients are finite.")
print()
print("✓ READY FOR TRAINING LOOP")
print("=" * 80)

RE-VERIFYING LOSS AFTER TARGET NORMALIZATION

--------------------------------------------------------------------------------
15min_5min
--------------------------------------------------------------------------------
Input shape:  (8, 4, 3600)
Target shape: (8, 3, 3600)

Target ranges:
  Baseline: 162.9432 to 177.9872
  ACC:      0.0000 to 1.0000
  DEC:      0.0000 to 1.0000

  ACC unique values: [0. 1.]
  DEC unique values: [0. 1.]

Loss:
  Total:    172.665604
  Baseline: 171.016556
  ACC:      0.792121
  DEC:      0.856928

✓ Forward pass successful.
✓ Loss is finite.
✓ Backward pass successful.
✓ Gradients are finite.

✓ LOSS VERIFICATION COMPLETED
✓ Baseline uses SmoothL1Loss.
✓ ACC uses BCEWithLogitsLoss with 0/1 targets.
✓ DEC uses BCEWithLogitsLoss with 0/1 targets.
✓ Forward pass is valid.
✓ Backward pass is valid.
✓ Gradients are finite.

✓ READY FOR TRAINING LOOP


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — AUC / PR-AUC / SENSITIVITY / PRECISION / F1
#
# Best model is selected by:
#     0.5 * ACC PR-AUC + 0.5 * DEC PR-AUC
#
# ACC / DEC targets are already normalized to 0/1.
# Class-weighted BCE is preserved:
#     ACC pos_weight = 10.0
#     DEC pos_weight = 1.5
# ══════════════════════════════════════════════════════════════════════════════

import copy
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)


# ══════════════════════════════════════════════════════════════════════════════
# 1. TRAINING SETTINGS
# ══════════════════════════════════════════════════════════════════════════════

NUM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
GRADIENT_CLIP_NORM = 1.0


TRAINED_MODELS = {}
TRAINING_HISTORY = {}
BEST_VALIDATION_LOSS = {}
BEST_EPOCHS = {}


# ══════════════════════════════════════════════════════════════════════════════
# 2. TRAIN ONE EXPERIMENT
# ══════════════════════════════════════════════════════════════════════════════

def train_experiment(
    experiment_name,
    train_loader,
    val_loader,
    num_epochs=15,
    early_stopping_patience=3,
    gradient_clip_norm=1.0
):

    print()
    print("=" * 80)
    print(f"TRAINING EXPERIMENT: {experiment_name}")
    print("=" * 80)


    # ------------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------------

    experiment_model = CTGUNet1D(
        in_channels=4,
        out_channels=3
    ).to(device)


    # ------------------------------------------------------------------
    # CLASS-WEIGHTED LOSS
    #
    # ACC: pos_weight = 10.0
    # DEC: pos_weight = 1.5
    # ------------------------------------------------------------------

    experiment_criterion = CTGMultiTaskLoss(
        baseline_weight=1.0,
        acc_weight=1.0,
        dec_weight=1.0,
        acc_pos_weight=ACC_POS_WEIGHT,
        dec_pos_weight=DEC_POS_WEIGHT
    )


    # ------------------------------------------------------------------
    # OPTIMIZER
    # ------------------------------------------------------------------

    experiment_optimizer = torch.optim.AdamW(
        experiment_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    # ------------------------------------------------------------------
    # LEARNING-RATE SCHEDULER
    # ------------------------------------------------------------------

    experiment_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        experiment_optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )


    # ------------------------------------------------------------------
    # BEST CHECKPOINT TRACKING
    #
    # Best model = highest combined PR-AUC.
    # ------------------------------------------------------------------

    best_checkpoint_score = -float("inf")

    best_epoch = 0

    best_val_loss = float("inf")

    epochs_without_improvement = 0

    best_state_dict = None


    # ------------------------------------------------------------------
    # TRAINING HISTORY
    # ------------------------------------------------------------------

    history = {

        "train_loss": [],
        "val_loss": [],

        "acc_auc": [],
        "acc_pr_auc": [],
        "acc_sensitivity": [],
        "acc_precision": [],
        "acc_f1": [],

        "dec_auc": [],
        "dec_pr_auc": [],
        "dec_sensitivity": [],
        "dec_precision": [],
        "dec_f1": [],

    }


    # ══════════════════════════════════════════════════════════════════
    # EPOCH LOOP
    # ══════════════════════════════════════════════════════════════════

    for epoch in range(
        1,
        num_epochs + 1
    ):


        # ==============================================================
        # TRAIN
        # ==============================================================

        experiment_model.train()

        train_loss_sum = 0.0
        train_batches = 0


        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)


            experiment_optimizer.zero_grad(
                set_to_none=True
            )


            prediction = experiment_model(x)


            loss, components = experiment_criterion(
                prediction,
                y
            )


            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite loss in "
                    f"{experiment_name}, "
                    f"epoch {epoch}"
                )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                experiment_model.parameters(),
                max_norm=gradient_clip_norm
            )


            experiment_optimizer.step()


            train_loss_sum += loss.item()

            train_batches += 1


        if train_batches == 0:

            raise RuntimeError(
                f"No training batches available "
                f"for {experiment_name}"
            )


        train_loss = (
            train_loss_sum /
            train_batches
        )


        # ==============================================================
        # VALIDATION
        # ==============================================================

        experiment_model.eval()


        val_loss_sum = 0.0
        val_batches = 0


        acc_targets = []
        acc_probs = []

        dec_targets = []
        dec_probs = []


        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(device)
                y = y.to(device)


                prediction = experiment_model(x)


                loss, components = experiment_criterion(
                    prediction,
                    y
                )


                if not torch.isfinite(loss):

                    raise RuntimeError(
                        f"Non-finite validation loss "
                        f"in {experiment_name}, "
                        f"epoch {epoch}"
                    )


                val_loss_sum += loss.item()

                val_batches += 1


                # ------------------------------------------------------
                # Convert logits -> probabilities
                # ------------------------------------------------------

                acc_prob = torch.sigmoid(
                    prediction[:, 1, :]
                ).cpu().numpy()


                dec_prob = torch.sigmoid(
                    prediction[:, 2, :]
                ).cpu().numpy()


                acc_target = (
                    y[:, 1, :]
                    .cpu()
                    .numpy()
                )


                dec_target = (
                    y[:, 2, :]
                    .cpu()
                    .numpy()
                )


                acc_probs.append(
                    acc_prob.reshape(-1)
                )

                acc_targets.append(
                    acc_target.reshape(-1)
                )


                dec_probs.append(
                    dec_prob.reshape(-1)
                )

                dec_targets.append(
                    dec_target.reshape(-1)
                )


        if val_batches == 0:

            raise RuntimeError(
                f"No validation batches available "
                f"for {experiment_name}"
            )


        val_loss = (
            val_loss_sum /
            val_batches
        )


        acc_probs = np.concatenate(
            acc_probs
        )

        acc_targets = np.concatenate(
            acc_targets
        )


        dec_probs = np.concatenate(
            dec_probs
        )

        dec_targets = np.concatenate(
            dec_targets
        )


        # ==============================================================
        # VERIFY TARGET ENCODING
        # ==============================================================

        if not np.all(
            np.isin(
                acc_targets,
                [0.0, 1.0]
            )
        ):

            raise RuntimeError(
                f"Invalid ACC targets detected "
                f"during validation in "
                f"{experiment_name}, "
                f"epoch {epoch}"
            )


        if not np.all(
            np.isin(
                dec_targets,
                [0.0, 1.0]
            )
        ):

            raise RuntimeError(
                f"Invalid DEC targets detected "
                f"during validation in "
                f"{experiment_name}, "
                f"epoch {epoch}"
            )


        # ==============================================================
        # CLASSIFICATION PREDICTIONS
        # ==============================================================

        acc_pred = (
            acc_probs >= 0.50
        ).astype(int)


        dec_pred = (
            dec_probs >= 0.50
        ).astype(int)


        # ==============================================================
        # ACC METRICS
        # ==============================================================

        acc_auc = roc_auc_score(
            acc_targets,
            acc_probs
        )


        acc_pr_auc = average_precision_score(
            acc_targets,
            acc_probs
        )


        acc_sensitivity = recall_score(
            acc_targets,
            acc_pred,
            zero_division=0
        )


        acc_precision = precision_score(
            acc_targets,
            acc_pred,
            zero_division=0
        )


        acc_f1 = f1_score(
            acc_targets,
            acc_pred,
            zero_division=0
        )


        # ==============================================================
        # DEC METRICS
        # ==============================================================

        dec_auc = roc_auc_score(
            dec_targets,
            dec_probs
        )


        dec_pr_auc = average_precision_score(
            dec_targets,
            dec_probs
        )


        dec_sensitivity = recall_score(
            dec_targets,
            dec_pred,
            zero_division=0
        )


        dec_precision = precision_score(
            dec_targets,
            dec_pred,
            zero_division=0
        )


        dec_f1 = f1_score(
            dec_targets,
            dec_pred,
            zero_division=0
        )


        # ==============================================================
        # LEARNING-RATE SCHEDULER
        # ==============================================================

        experiment_scheduler.step(
            val_loss
        )


        current_lr = (
            experiment_optimizer
            .param_groups[0]["lr"]
        )


        # ==============================================================
        # SAVE HISTORY
        # ==============================================================

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )


        history["acc_auc"].append(
            acc_auc
        )

        history["acc_pr_auc"].append(
            acc_pr_auc
        )

        history["acc_sensitivity"].append(
            acc_sensitivity
        )

        history["acc_precision"].append(
            acc_precision
        )

        history["acc_f1"].append(
            acc_f1
        )


        history["dec_auc"].append(
            dec_auc
        )

        history["dec_pr_auc"].append(
            dec_pr_auc
        )

        history["dec_sensitivity"].append(
            dec_sensitivity
        )

        history["dec_precision"].append(
            dec_precision
        )

        history["dec_f1"].append(
            dec_f1
        )


        # ==============================================================
        # COMBINED PR-AUC CHECKPOINT SCORE
        # ==============================================================

        checkpoint_score = (

            0.5 * acc_pr_auc

            + 0.5 * dec_pr_auc

        )


        # ==============================================================
        # PRINT RESULTS
        # ==============================================================

        print()

        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Train {train_loss:.3f} | "
            f"Val {val_loss:.3f} | "
            f"LR {current_lr:.2g}"
        )


        print(
            f"  ACC | "
            f"AUC {acc_auc:.3f} | "
            f"PR-AUC {acc_pr_auc:.3f} | "
            f"Sens {acc_sensitivity:.3f} | "
            f"Prec {acc_precision:.3f} | "
            f"F1 {acc_f1:.3f}"
        )


        print(
            f"  DEC | "
            f"AUC {dec_auc:.3f} | "
            f"PR-AUC {dec_pr_auc:.3f} | "
            f"Sens {dec_sensitivity:.3f} | "
            f"Prec {dec_precision:.3f} | "
            f"F1 {dec_f1:.3f}"
        )


        print(
            f"  Combined PR-AUC: "
            f"{checkpoint_score:.3f}"
        )


        # ==============================================================
        # SAVE BEST MODEL
        # ==============================================================

        if checkpoint_score > best_checkpoint_score:

            best_checkpoint_score = (
                checkpoint_score
            )

            best_epoch = epoch

            best_val_loss = val_loss

            epochs_without_improvement = 0


            best_state_dict = copy.deepcopy(
                experiment_model.state_dict()
            )


            print(
                f"  ✓ BEST MODEL "
                f"(combined PR-AUC = "
                f"{checkpoint_score:.3f})"
            )


        else:

            epochs_without_improvement += 1


        # ==============================================================
        # EARLY STOPPING
        # ==============================================================

        if (
            epochs_without_improvement
            >= early_stopping_patience
        ):

            print()

            print(
                f"Early stopping at epoch "
                f"{epoch}. "
                f"Best epoch: {best_epoch}"
            )

            break


    # ══════════════════════════════════════════════════════════════════
    # RESTORE BEST MODEL
    # ══════════════════════════════════════════════════════════════════

    if best_state_dict is None:

        raise RuntimeError(
            f"No valid best model was saved "
            f"for {experiment_name}"
        )


    experiment_model.load_state_dict(
        best_state_dict
    )

    experiment_model.eval()


    # ══════════════════════════════════════════════════════════════════
    # FINAL REPORT
    # ══════════════════════════════════════════════════════════════════

    print()

    print("-" * 80)

    print(
        f"COMPLETED: {experiment_name}"
    )

    print(
        f"Best epoch: "
        f"{best_epoch}"
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.4f}"
    )

    print(
        f"Best combined PR-AUC: "
        f"{best_checkpoint_score:.4f}"
    )

    print("-" * 80)


    return (
        experiment_model,
        history,
        best_val_loss,
        best_epoch
    )


# ══════════════════════════════════════════════════════════════════════════════
# 3. TRAIN ALL EXPERIMENTS
# ══════════════════════════════════════════════════════════════════════════════

for experiment_name in EXPERIMENTS:

    train_loader = (
        DATALOADERS[
            experiment_name
        ]["train"]
    )

    val_loader = (
        DATALOADERS[
            experiment_name
        ]["val"]
    )


    (
        trained_model,
        history,
        best_val_loss,
        best_epoch
    ) = train_experiment(

        experiment_name=experiment_name,

        train_loader=train_loader,

        val_loader=val_loader,

        num_epochs=NUM_EPOCHS,

        early_stopping_patience=(
            EARLY_STOPPING_PATIENCE
        ),

        gradient_clip_norm=(
            GRADIENT_CLIP_NORM
        )
    )


    TRAINED_MODELS[
        experiment_name
    ] = trained_model


    TRAINING_HISTORY[
        experiment_name
    ] = history


    BEST_VALIDATION_LOSS[
        experiment_name
    ] = best_val_loss


    BEST_EPOCHS[
        experiment_name
    ] = best_epoch


# ══════════════════════════════════════════════════════════════════════════════
# 4. FINAL TRAINING REPORT
# ══════════════════════════════════════════════════════════════════════════════

print()

print("=" * 80)
print("✓ TRAINING COMPLETED")
print("=" * 80)


for experiment_name in EXPERIMENTS:

    print()

    print(experiment_name)

    print(
        f"  Best epoch: "
        f"{BEST_EPOCHS[experiment_name]}"
    )

    print(
        f"  Best validation loss: "
        f"{BEST_VALIDATION_LOSS[experiment_name]:.4f}"
    )

    final_history = (
        TRAINING_HISTORY[
            experiment_name
        ]
    )

    best_epoch_index = (
        BEST_EPOCHS[experiment_name] - 1
    )

    print(
        f"  Best ACC PR-AUC: "
        f"{final_history['acc_pr_auc'][best_epoch_index]:.4f}"
    )

    print(
        f"  Best DEC PR-AUC: "
        f"{final_history['dec_pr_auc'][best_epoch_index]:.4f}"
    )


print()
print("✓ Best model selected using combined ACC/DEC PR-AUC.")
print("✓ Class-weighted BCE preserved.")
print("✓ ACC/DEC targets are 0/1.")
print("✓ Early stopping enabled.")
print("✓ Best model restored after training.")
print("=" * 80)


TRAINING EXPERIMENT: 15min_5min

Epoch 01/5 | Train 170.884 | Val 170.208 | LR 0.0003
  ACC | AUC 1.000 | PR-AUC 1.000 | Sens 1.000 | Prec 0.869 | F1 0.930
  DEC | AUC 0.993 | PR-AUC 0.992 | Sens 0.949 | Prec 0.935 | F1 0.942
  Combined PR-AUC: 0.996
  ✓ BEST MODEL (combined PR-AUC = 0.996)

Epoch 02/5 | Train 169.913 | Val 169.497 | LR 0.0003
  ACC | AUC 1.000 | PR-AUC 1.000 | Sens 1.000 | Prec 0.951 | F1 0.975
  DEC | AUC 1.000 | PR-AUC 1.000 | Sens 0.987 | Prec 0.992 | F1 0.990
  Combined PR-AUC: 1.000
  ✓ BEST MODEL (combined PR-AUC = 1.000)

Epoch 03/5 | Train 169.202 | Val 168.882 | LR 0.0003
  ACC | AUC 1.000 | PR-AUC 1.000 | Sens 1.000 | Prec 0.987 | F1 0.993
  DEC | AUC 0.998 | PR-AUC 0.998 | Sens 0.985 | Prec 0.965 | F1 0.975
  Combined PR-AUC: 0.999

Epoch 04/5 | Train 168.499 | Val 168.139 | LR 0.0003
  ACC | AUC 1.000 | PR-AUC 1.000 | Sens 1.000 | Prec 0.997 | F1 0.998
  DEC | AUC 1.000 | PR-AUC 1.000 | Sens 0.997 | Prec 0.997 | F1 0.997
  Combined PR-AUC: 1.000
  ✓ BEST 

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL MODEL EVALUATION
# AUC / PR-AUC / SENSITIVITY / SPECIFICITY / PRECISION / F1
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print()
print("=" * 80)
print("FINAL VALIDATION PERFORMANCE")
print("=" * 80)


def evaluate_model(experiment_name, model, val_loader):

    model.eval()

    acc_targets = []
    acc_probs = []

    dec_targets = []
    dec_probs = []

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)
            y = y.to(device)

            prediction = model(x)

            acc_prob = torch.sigmoid(
                prediction[:, 1, :]
            ).cpu().numpy().reshape(-1)

            dec_prob = torch.sigmoid(
                prediction[:, 2, :]
            ).cpu().numpy().reshape(-1)

            acc_target = (
                y[:, 1, :]
                .cpu()
                .numpy()
                .reshape(-1)
            )

            dec_target = (
                y[:, 2, :]
                .cpu()
                .numpy()
                .reshape(-1)
            )

            acc_probs.append(acc_prob)
            acc_targets.append(acc_target)

            dec_probs.append(dec_prob)
            dec_targets.append(dec_target)

    acc_probs = np.concatenate(acc_probs)
    acc_targets = np.concatenate(acc_targets)

    dec_probs = np.concatenate(dec_probs)
    dec_targets = np.concatenate(dec_targets)


    # ==============================================================
    # THRESHOLD
    # ==============================================================

    threshold = 0.50

    acc_pred = (acc_probs >= threshold).astype(int)
    dec_pred = (dec_probs >= threshold).astype(int)


    # ==============================================================
    # METRICS
    # ==============================================================

    results = {}

    for name, targets, probs, preds in [
        ("ACC", acc_targets, acc_probs, acc_pred),
        ("DEC", dec_targets, dec_probs, dec_pred)
    ]:

        tn, fp, fn, tp = confusion_matrix(
            targets,
            preds,
            labels=[0, 1]
        ).ravel()

        sensitivity = recall_score(
            targets,
            preds,
            zero_division=0
        )

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0.0
        )

        precision = precision_score(
            targets,
            preds,
            zero_division=0
        )

        f1 = f1_score(
            targets,
            preds,
            zero_division=0
        )

        auc = roc_auc_score(
            targets,
            probs
        )

        pr_auc = average_precision_score(
            targets,
            probs
        )

        results[name] = {
            "auc": auc,
            "pr_auc": pr_auc,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "f1": f1,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "prevalence": targets.mean()
        }


    # ==============================================================
    # PRINT
    # ==============================================================

    print()
    print(f"EXPERIMENT: {experiment_name}")
    print("-" * 80)

    for name in ["ACC", "DEC"]:

        r = results[name]

        print()
        print(name)

        print(
            f"  Prevalence:   {r['prevalence']:.4%}"
        )

        print(
            f"  ROC-AUC:      {r['auc']:.4f}"
        )

        print(
            f"  PR-AUC:       {r['pr_auc']:.4f}"
        )

        print(
            f"  Sensitivity:   {r['sensitivity']:.4f}"
        )

        print(
            f"  Specificity:   {r['specificity']:.4f}"
        )

        print(
            f"  Precision:     {r['precision']:.4f}"
        )

        print(
            f"  F1:            {r['f1']:.4f}"
        )

        print()
        print(
            f"  TP: {r['tp']:,} | "
            f"TN: {r['tn']:,} | "
            f"FP: {r['fp']:,} | "
            f"FN: {r['fn']:,}"
        )

    return results


# ══════════════════════════════════════════════════════════════════════════════
# EVALUATE ALL EXPERIMENTS
# ══════════════════════════════════════════════════════════════════════════════

FINAL_RESULTS = {}

for experiment_name in EXPERIMENTS:

    FINAL_RESULTS[experiment_name] = evaluate_model(
        experiment_name=experiment_name,
        model=TRAINED_MODELS[experiment_name],
        val_loader=DATALOADERS[experiment_name]["val"]
    )


print()
print("=" * 80)
print("✓ FINAL EVALUATION COMPLETED")
print("=" * 80)


FINAL VALIDATION PERFORMANCE

EXPERIMENT: 15min_5min
--------------------------------------------------------------------------------

ACC
  Prevalence:   3.6310%
  ROC-AUC:      1.0000
  PR-AUC:       1.0000
  Sensitivity:   1.0000
  Specificity:   0.9999
  Precision:     0.9967
  F1:            0.9984

  TP: 24,705 | TN: 655,614 | FP: 81 | FN: 0

DEC
  Prevalence:   44.0833%
  ROC-AUC:      1.0000
  PR-AUC:       1.0000
  Sensitivity:   0.9970
  Specificity:   0.9977
  Precision:     0.9970
  F1:            0.9970

  TP: 299,052 | TN: 379,566 | FP: 891 | FN: 891

✓ FINAL EVALUATION COMPLETED


In [15]:
# =============================================================================
# FINAL: FREEZE + SAVE PRETRAINED ACC/DEC DETECTORS DIRECTLY TO GOOGLE DRIVE
# =============================================================================

import os
import torch
from google.colab import drive


# -------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# -------------------------------------------------------------------------

drive.mount("/content/drive")


# -------------------------------------------------------------------------
# 2. PERMANENT DRIVE LOCATION
# -------------------------------------------------------------------------

PRETRAINED_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Fetal_project"
)

os.makedirs(
    PRETRAINED_DIR,
    exist_ok=True
)


# -------------------------------------------------------------------------
# 3. VERIFY TRAINED MODELS EXIST
# -------------------------------------------------------------------------

if not TRAINED_MODELS:

    raise RuntimeError(
        "TRAINED_MODELS is empty. "
        "Training must complete successfully before saving."
    )


print("=" * 80)
print("SAVING PRETRAINED ACC/DEC DETECTORS")
print("=" * 80)


# -------------------------------------------------------------------------
# 4. FREEZE + SAVE ALL TRAINED DETECTORS
# -------------------------------------------------------------------------

for experiment_name, trained_model in TRAINED_MODELS.items():

    print()
    print("-" * 80)
    print(f"EXPERIMENT: {experiment_name}")
    print("-" * 80)


    # ---------------------------------------------------------------------
    # Freeze model
    # ---------------------------------------------------------------------

    for parameter in trained_model.parameters():

        parameter.requires_grad = False


    trained_model.eval()


    # ---------------------------------------------------------------------
    # Retrieve best epoch
    # ---------------------------------------------------------------------

    best_epoch = BEST_EPOCHS.get(
        experiment_name,
        None
    )


    # ---------------------------------------------------------------------
    # Retrieve best combined PR-AUC
    #
    # The training loop selects the checkpoint using:
    #
    #     0.5 * ACC PR-AUC
    #     +
    #     0.5 * DEC PR-AUC
    # ---------------------------------------------------------------------

    history = TRAINING_HISTORY.get(
        experiment_name,
        {}
    )


    acc_pr_auc_history = history.get(
        "acc_pr_auc",
        []
    )

    dec_pr_auc_history = history.get(
        "dec_pr_auc",
        []
    )


    if (
        len(acc_pr_auc_history) > 0
        and
        len(dec_pr_auc_history) > 0
    ):

        combined_pr_auc_history = [

            0.5 * acc_pr_auc
            +
            0.5 * dec_pr_auc

            for acc_pr_auc, dec_pr_auc
            in zip(
                acc_pr_auc_history,
                dec_pr_auc_history
            )
        ]


        best_combined_pr_auc = max(
            combined_pr_auc_history
        )

    else:

        best_combined_pr_auc = None


    # ---------------------------------------------------------------------
    # Checkpoint
    # ---------------------------------------------------------------------

    checkpoint = {

        "model_state_dict":
            trained_model.state_dict(),

        "experiment_name":
            experiment_name,

        "in_channels":
            4,

        "out_channels":
            3,

        "best_epoch":
            best_epoch,

        "best_combined_pr_auc":
            best_combined_pr_auc,

        "frozen":
            True,

        "description":
            "Pretrained ACC/DEC temporal detector",

        "target_encoding":
            "ACC/DEC encoded as 0/1",

        "acc_pos_weight":
            10.0,

        "dec_pos_weight":
            1.5
    }


    # ---------------------------------------------------------------------
    # Filename
    # ---------------------------------------------------------------------

    filename = (
        f"{experiment_name}_acc_dec_detector.pt"
    )


    path = os.path.join(
        PRETRAINED_DIR,
        filename
    )


    # ---------------------------------------------------------------------
    # SAVE DIRECTLY TO GOOGLE DRIVE
    # ---------------------------------------------------------------------

    torch.save(
        checkpoint,
        path
    )


    # ---------------------------------------------------------------------
    # VERIFY FILE EXISTS
    # ---------------------------------------------------------------------

    if not os.path.isfile(path):

        raise RuntimeError(
            f"FAILED TO SAVE: {path}"
        )


    size_mb = (
        os.path.getsize(path)
        /
        (1024 ** 2)
    )


    print(
        f"✓ SAVED: {filename}"
    )

    print(
        f"  Location: {path}"
    )

    print(
        f"  Size: {size_mb:.2f} MB"
    )

    print(
        f"  Best epoch: {best_epoch}"
    )


    if best_combined_pr_auc is not None:

        print(
            f"  Best combined PR-AUC: "
            f"{best_combined_pr_auc:.6f}"
        )

    else:

        print(
            "  Best combined PR-AUC: "
            "not available"
        )


    print(
        "  Frozen: True"
    )


# -------------------------------------------------------------------------
# 5. FINAL GOOGLE DRIVE VERIFICATION
# -------------------------------------------------------------------------

print()
print("=" * 80)
print("VERIFYING GOOGLE DRIVE FILES")
print("=" * 80)


for experiment_name in TRAINED_MODELS:

    filename = (
        f"{experiment_name}_acc_dec_detector.pt"
    )


    path = os.path.join(
        PRETRAINED_DIR,
        filename
    )


    # ---------------------------------------------------------------------
    # Verify file exists
    # ---------------------------------------------------------------------

    assert os.path.isfile(path), (
        f"Missing checkpoint: {path}"
    )


    # ---------------------------------------------------------------------
    # Verify checkpoint can actually be loaded
    #
    # PyTorch 2.6+ defaults to weights_only=True.
    #
    # This checkpoint was created by this notebook and is trusted,
    # so weights_only=False is appropriate here.
    # ---------------------------------------------------------------------

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )


    # ---------------------------------------------------------------------
    # Verify checkpoint contents
    # ---------------------------------------------------------------------

    assert (
        "model_state_dict"
        in checkpoint
    ), (
        f"Missing model_state_dict: {filename}"
    )


    assert (
        checkpoint["frozen"]
        is True
    ), (
        f"Checkpoint is not marked frozen: {filename}"
    )


    assert (
        checkpoint["experiment_name"]
        == experiment_name
    ), (
        f"Experiment name mismatch: {filename}"
    )


    assert (
        checkpoint["in_channels"]
        == 4
    ), (
        f"Incorrect input channels: {filename}"
    )


    assert (
        checkpoint["out_channels"]
        == 3
    ), (
        f"Incorrect output channels: {filename}"
    )


    print(
        f"✓ {filename}"
    )

    print(
        f"  Checkpoint loaded successfully."
    )

    print(
        f"  Model state verified."
    )

    print(
        f"  Frozen flag verified."
    )


# -------------------------------------------------------------------------
# 6. FINAL CONFIRMATION
# -------------------------------------------------------------------------

print()
print("=" * 80)
print("✓ PRETRAINED DETECTORS FROZEN")
print("✓ PRETRAINED DETECTORS SAVED TO GOOGLE DRIVE")
print("✓ CHECKPOINTS SUCCESSFULLY LOADED")
print("✓ CHECKPOINT CONTENTS VERIFIED")
print("=" * 80)

print()
print("Google Drive folder:")
print(PRETRAINED_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAVING PRETRAINED ACC/DEC DETECTORS

--------------------------------------------------------------------------------
EXPERIMENT: 15min_5min
--------------------------------------------------------------------------------
✓ SAVED: 15min_5min_acc_dec_detector.pt
  Location: /content/drive/MyDrive/pretrained_detectors/15min_5min_acc_dec_detector.pt
  Size: 4.10 MB
  Best epoch: 4
  Best combined PR-AUC: 0.999986
  Frozen: True

VERIFYING GOOGLE DRIVE FILES
✓ 15min_5min_acc_dec_detector.pt
  Checkpoint loaded successfully.
  Model state verified.
  Frozen flag verified.

✓ PRETRAINED DETECTORS FROZEN
✓ PRETRAINED DETECTORS SAVED TO GOOGLE DRIVE
✓ CHECKPOINTS SUCCESSFULLY LOADED
✓ CHECKPOINT CONTENTS VERIFIED

Google Drive folder:
/content/drive/MyDrive/pretrained_detectors
